In [1]:
!pip install pandas scikit-learn matplotlib pyarrow

In [2]:
import numpy as np
import random
import pandas as pd
from pathlib import Path

np.random.seed(42)
random.seed(42)

In [3]:
project_root = Path(".")
player_path = project_root / "training_dataset.parquet"
matchup_path = project_root / "tournament_matchups.jsonl"
bracket_path = project_root / "bracket_2026.json"

output_dir = project_root / "outputs"
output_dir.mkdir(exist_ok=True)

print("Current folder:", project_root.resolve())

Current folder: /Users/yahiaali/Final Project/Yahia Ali - Final Project


In [4]:
from src.data_utils import (
    clean_player_data,
    load_bracket,
    load_matchups,
    load_player_data,
)

from src.features import (
    build_matchup_dataset,
    build_team_features,
    get_xy,
    split_by_season,
)

from src.modeling import (
    build_model,
    evaluate_model,
    run_kfold_cv,
)

In [5]:
print("Loading data...")
players = load_player_data(player_path)
matchups = load_matchups(matchup_path)
bracket = load_bracket(bracket_path)

print("Players shape:", players.shape)
print("Matchups shape:", matchups.shape)

Loading data...
Players shape: (13299, 55)
Matchups shape: (917, 14)


In [6]:
print("Cleaning player data...")
players = clean_player_data(players)

print("Building team features...")
team_features = build_team_features(players)

print("Team features shape:", team_features.shape)

Cleaning player data...
Building team features...
Team features shape: (964, 48)


In [7]:
historical_games = matchups.loc[matchups["season_year"] <= 2025].copy()

print("Building historical matchup dataset...")
modeled_games, feature_cols = build_matchup_dataset(
    historical_games,
    team_features,
    include_target=True,
    augment_mirror=True,
)

print("Modeled games shape:", modeled_games.shape)
print("Number of features:", len(feature_cols))

Building historical matchup dataset...
Modeled games shape: (1762, 56)
Number of features: 46


In [8]:
train_df, val_df, test_df = split_by_season(
    modeled_games,
    train_end_year=2023,
    val_year=2024,
    test_year=2025,
)

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

Train rows: 1510
Validation rows: 126
Test rows: 126


In [9]:
X_train, y_train = get_xy(train_df, feature_cols)
X_val, y_val = get_xy(val_df, feature_cols)
X_test, y_test = get_xy(test_df, feature_cols)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1510, 46)
X_val shape: (126, 46)
X_test shape: (126, 46)


In [10]:
print("Running 5-fold CV...")
cv_results = run_kfold_cv(X_train, y_train, n_splits=5)

best_cv = cv_results[0]
print("Best CV result:")
print(best_cv)

Running 5-fold CV...
Best CV result:
{'model_name': 'linear_svm', 'params': {'C': 1.0}, 'cv_scores': [0.8741721854304636, 0.8576158940397351, 0.8774834437086093, 0.8112582781456954, 0.8973509933774835], 'cv_mean_accuracy': 0.8635761589403973, 'cv_std_accuracy': 0.029048625429750077}


In [11]:
best_model = build_model(best_cv["model_name"], best_cv["params"])
best_model.fit(X_train, y_train)

val_metrics = evaluate_model(best_model, X_val, y_val)
test_metrics = evaluate_model(best_model, X_test, y_test)

print("Validation Accuracy:", round(val_metrics["accuracy"], 3))
print("Test Accuracy:", round(test_metrics["accuracy"], 3))

Validation Accuracy: 0.794
Test Accuracy: 0.889


In [12]:
print("Retraining final model on all historical games...")

X_final, y_final = get_xy(modeled_games, feature_cols)

final_model = build_model(best_cv["model_name"], best_cv["params"])
final_model.fit(X_final, y_final)

print("Final model ready.")

Retraining final model on all historical games...
Final model ready.


In [13]:
def predict_round(games_df, team_features, feature_cols, model):
    round_features, _ = build_matchup_dataset(
        games_df,
        team_features,
        include_target=False,
        augment_mirror=False,
    )

    keep_cols = ["season_year", "round", "region", "game_number"]
    round_features = round_features.merge(
        games_df[keep_cols + ["team_a_slug", "team_b_slug"]],
        on=["season_year", "round", "region", "team_a_slug", "team_b_slug"],
        how="left",
    )

    X_round = round_features[feature_cols].copy()

    round_features["predicted_label"] = model.predict(X_round)
    round_features["team_a_win_probability"] = model.predict_proba(X_round)[:, 1]

    round_features["predicted_winner_slug"] = round_features.apply(
        lambda row: row["team_a_slug"] if row["predicted_label"] == 1 else row["team_b_slug"],
        axis=1,
    )

    round_features["predicted_winner_name"] = round_features.apply(
        lambda row: row["team_a_name"] if row["predicted_label"] == 1 else row["team_b_name"],
        axis=1,
    )

    round_features["predicted_winner_seed"] = round_features.apply(
        lambda row: row["team_a_seed"] if row["predicted_label"] == 1 else row["team_b_seed"],
        axis=1,
    )

    return round_features.sort_values(["region", "game_number"]).reset_index(drop=True)

In [14]:
def build_next_round_from_winners(winners_df, next_round_name):
    winners_df = winners_df.copy().sort_values(["region", "game_number"]).reset_index(drop=True)

    next_games = []

    for region in winners_df["region"].unique():
        region_df = winners_df[winners_df["region"] == region].copy()
        region_df = region_df.sort_values("game_number").reset_index(drop=True)

        region_df["next_game_number"] = region_df["game_number"] // 2

        for next_game_number, group in region_df.groupby("next_game_number", sort=True):
            group = group.sort_values("game_number").reset_index(drop=True)

            team1 = group.iloc[0]
            team2 = group.iloc[1]

            next_games.append({
                "season_year": int(team1["season_year"]),
                "round": next_round_name,
                "region": region,
                "game_number": int(next_game_number),
                "team_a_slug": team1["predicted_winner_slug"],
                "team_a_name": team1["predicted_winner_name"],
                "team_a_seed": int(team1["predicted_winner_seed"]),
                "team_b_slug": team2["predicted_winner_slug"],
                "team_b_name": team2["predicted_winner_name"],
                "team_b_seed": int(team2["predicted_winner_seed"]),
            })

    return pd.DataFrame(next_games).sort_values(["region", "game_number"]).reset_index(drop=True)

In [15]:
R64_ORDER = {
    (1, 16): 0,
    (8, 9): 1,
    (5, 12): 2,
    (4, 13): 3,
    (6, 11): 4,
    (3, 14): 5,
    (7, 10): 6,
    (2, 15): 7,
}

def assign_r64_game_numbers(r64_region_games):
    r64_region_games = r64_region_games.copy()

    def get_game_number(row):
        seed_pair = (int(row["team_a_seed"]), int(row["team_b_seed"]))
        if seed_pair not in R64_ORDER:
            raise ValueError(f"Unexpected R64 seed pair: {seed_pair}")
        return R64_ORDER[seed_pair]

    r64_region_games["game_number"] = r64_region_games.apply(get_game_number, axis=1)
    return r64_region_games.sort_values("game_number").reset_index(drop=True)

In [16]:
def simulate_region(r64_region_games, team_features, feature_cols, model):
    r64_region_games = assign_r64_game_numbers(r64_region_games)

    r64_pred = predict_round(r64_region_games, team_features, feature_cols, model)

    r32_games = build_next_round_from_winners(r64_pred, "R32")
    r32_pred = predict_round(r32_games, team_features, feature_cols, model)

    s16_games = build_next_round_from_winners(r32_pred, "S16")
    s16_pred = predict_round(s16_games, team_features, feature_cols, model)

    e8_games = build_next_round_from_winners(s16_pred, "E8")
    e8_pred = predict_round(e8_games, team_features, feature_cols, model)

    return {
        "R64": r64_pred,
        "R32": r32_pred,
        "S16": s16_pred,
        "E8": e8_pred,
        "region_winner": e8_pred.iloc[0],
    }

In [17]:
games_2026 = matchups.loc[matchups["season_year"] == 2026].copy()

first_four_games = games_2026.loc[games_2026["round"] == "FirstFour"].copy()
r64_games = games_2026.loc[games_2026["round"] == "R64"].copy()

first_four_games = first_four_games.reset_index(drop=True)
first_four_games["game_number"] = range(len(first_four_games))

print("First Four games:", len(first_four_games))
print("Round of 64 games:", len(r64_games))

First Four games: 4
Round of 64 games: 32


In [18]:
first_four_pred = predict_round(first_four_games, team_features, feature_cols, final_model)

In [19]:
first_four_map = {}

for _, row in first_four_pred.iterrows():
    key = f"FF_WINNER_{row['region']}_{int(row['team_a_seed'])}"
    first_four_map[key] = {
        "slug": row["predicted_winner_slug"],
        "name": row["predicted_winner_name"],
        "seed": int(row["predicted_winner_seed"]),
    }

def replace_placeholder_team(slug, name, seed):
    if isinstance(slug, str) and slug in first_four_map:
        repl = first_four_map[slug]
        return repl["slug"], repl["name"], repl["seed"]
    return slug, name, seed

r64_games = r64_games.copy()

r64_games[["team_a_slug", "team_a_name", "team_a_seed"]] = r64_games.apply(
    lambda row: pd.Series(
        replace_placeholder_team(row["team_a_slug"], row["team_a_name"], row["team_a_seed"])
    ),
    axis=1
)

r64_games[["team_b_slug", "team_b_name", "team_b_seed"]] = r64_games.apply(
    lambda row: pd.Series(
        replace_placeholder_team(row["team_b_slug"], row["team_b_name"], row["team_b_seed"])
    ),
    axis=1
)

print("Round of 64 placeholders replaced.")

Round of 64 placeholders replaced.


In [20]:
east = simulate_region(
    r64_games[r64_games["region"] == "East"].copy(),
    team_features,
    feature_cols,
    final_model,
)

south = simulate_region(
    r64_games[r64_games["region"] == "South"].copy(),
    team_features,
    feature_cols,
    final_model,
)

west = simulate_region(
    r64_games[r64_games["region"] == "West"].copy(),
    team_features,
    feature_cols,
    final_model,
)

midwest = simulate_region(
    r64_games[r64_games["region"] == "Midwest"].copy(),
    team_features,
    feature_cols,
    final_model,
)

print("Regional predictions complete.")

Regional predictions complete.


In [21]:
final_four_games = pd.DataFrame([
    {
        "season_year": 2026,
        "round": "F4",
        "region": "Final Four",
        "game_number": 0,
        "team_a_slug": east["region_winner"]["predicted_winner_slug"],
        "team_a_name": east["region_winner"]["predicted_winner_name"],
        "team_a_seed": int(east["region_winner"]["predicted_winner_seed"]),
        "team_b_slug": south["region_winner"]["predicted_winner_slug"],
        "team_b_name": south["region_winner"]["predicted_winner_name"],
        "team_b_seed": int(south["region_winner"]["predicted_winner_seed"]),
    },
    {
        "season_year": 2026,
        "round": "F4",
        "region": "Final Four",
        "game_number": 1,
        "team_a_slug": west["region_winner"]["predicted_winner_slug"],
        "team_a_name": west["region_winner"]["predicted_winner_name"],
        "team_a_seed": int(west["region_winner"]["predicted_winner_seed"]),
        "team_b_slug": midwest["region_winner"]["predicted_winner_slug"],
        "team_b_name": midwest["region_winner"]["predicted_winner_name"],
        "team_b_seed": int(midwest["region_winner"]["predicted_winner_seed"]),
    }
])

final_four_pred = predict_round(final_four_games, team_features, feature_cols, final_model)

In [22]:
championship_game = pd.DataFrame([
    {
        "season_year": 2026,
        "round": "NCG",
        "region": "Championship",
        "game_number": 0,
        "team_a_slug": final_four_pred.iloc[0]["predicted_winner_slug"],
        "team_a_name": final_four_pred.iloc[0]["predicted_winner_name"],
        "team_a_seed": int(final_four_pred.iloc[0]["predicted_winner_seed"]),
        "team_b_slug": final_four_pred.iloc[1]["predicted_winner_slug"],
        "team_b_name": final_four_pred.iloc[1]["predicted_winner_name"],
        "team_b_seed": int(final_four_pred.iloc[1]["predicted_winner_seed"]),
    }
])

championship_pred = predict_round(championship_game, team_features, feature_cols, final_model)

In [23]:
champion = championship_pred.iloc[0]["predicted_winner_name"]
print("Predicted Champion:", champion)

Predicted Champion: Michigan


In [24]:
all_rounds = pd.concat([
    first_four_pred,
    east["R64"], south["R64"], west["R64"], midwest["R64"],
    east["R32"], south["R32"], west["R32"], midwest["R32"],
    east["S16"], south["S16"], west["S16"], midwest["S16"],
    east["E8"], south["E8"], west["E8"], midwest["E8"],
    final_four_pred,
    championship_pred,
], ignore_index=True)

In [25]:
all_rounds_view = all_rounds[[
    "round",
    "region",
    "team_a_name",
    "team_a_seed",
    "team_b_name",
    "team_b_seed",
    "team_a_win_probability",
    "predicted_winner_name",
]].copy()

all_rounds_view

,round,region,team_a_name,team_a_seed,team_b_name,team_b_seed,team_a_win_probability,predicted_winner_name
0,FirstFour,Midwest,UMBC,16,Howard,16,0.196016,Howard
1,FirstFour,Midwest,Miami (OH),11,SMU,11,0.102581,SMU
2,FirstFour,South,Prairie View A&M,16,Lehigh,16,0.509884,Prairie View A&M
3,FirstFour,West,Texas,11,NC State,11,0.172909,NC State
4,R64,East,Duke,1,Siena,16,0.986657,Duke
...,...,...,...,...,...,...,...,...
62,E8,West,Arizona,1,Purdue,2,0.521935,Arizona
63,E8,Midwest,Michigan,1,Iowa State,2,0.744224,Michigan
64,F4,Final Four,UCLA,7,Houston,2,0.570998,UCLA
65,F4,Final Four,Arizona,1,Michigan,1,0.403861,Michigan


In [26]:
all_rounds_view.to_csv(output_dir / "full_tournament_predictions_2026.csv", index=False)

print("Saved to:", output_dir / "full_tournament_predictions_2026.csv")

Saved to: outputs/full_tournament_predictions_2026.csv


In [27]:
midwest["R64"][["game_number", "team_a_name", "team_b_name", "predicted_winner_name"]]

,game_number,team_a_name,team_b_name,predicted_winner_name
0,0,Michigan,Howard,Michigan
1,1,Georgia,Saint Louis,Saint Louis
2,2,Texas Tech,Akron,Texas Tech
3,3,Alabama,Hofstra,Hofstra
4,4,Tennessee,SMU,SMU
5,5,Virginia,Wright State,Virginia
6,6,Kentucky,Santa Clara,Santa Clara
7,7,Iowa State,Tennessee State,Iowa State


In [28]:
midwest["R32"][["game_number", "team_a_name", "team_b_name", "predicted_winner_name"]]

,game_number,team_a_name,team_b_name,predicted_winner_name
0,0,Michigan,Saint Louis,Michigan
1,1,Texas Tech,Hofstra,Texas Tech
2,2,SMU,Virginia,Virginia
3,3,Santa Clara,Iowa State,Iowa State


In [29]:
midwest["S16"][["game_number", "team_a_name", "team_b_name", "predicted_winner_name"]]

,game_number,team_a_name,team_b_name,predicted_winner_name
0,0,Michigan,Texas Tech,Michigan
1,1,Virginia,Iowa State,Iowa State
